<a href="https://colab.research.google.com/github/justunforgettable/AI-Cognitive-Ability-Predictor/blob/main/CogniScore_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

# For reproducibility
random.seed(42)
np.random.seed(42)

In [3]:
questions = [

{
    "id":1,
    "category":"Numerical",
    "difficulty":"Medium"
},

{
    "id":2,
    "category":"Numerical",
    "difficulty":"Hard"
},

{
    "id":3,
    "category":"Numerical",
    "difficulty":"Expert"
},

{
    "id":4,
    "category":"Logical",
    "difficulty":"Medium"
},

{
    "id":5,
    "category":"Logical",
    "difficulty":"Hard"
},

{
    "id":6,
    "category":"Logical",
    "difficulty":"Hard"
},

{
    "id":7,
    "category":"Pattern",
    "difficulty":"Medium"
},

{
    "id":8,
    "category":"Pattern",
    "difficulty":"Hard"
},

{
    "id":9,
    "category":"Pattern",
    "difficulty":"Expert"
},

{
    "id":10,
    "category":"Verbal",
    "difficulty":"Medium"
},

{
    "id":11,
    "category":"Verbal",
    "difficulty":"Hard"
},

{
    "id":12,
    "category":"Analytical",
    "difficulty":"Hard"
},

{
    "id":13,
    "category":"Analytical",
    "difficulty":"Expert"
},

{
    "id":14,
    "category":"Reflection",
    "difficulty":"Hard"
},

{
    "id":15,
    "category":"Reflection",
    "difficulty":"Medium"
}

]

In [4]:
def get_probability(ability, difficulty):

    if difficulty == "Medium":
        return min(0.95, ability + 0.15)

    elif difficulty == "Hard":
        return max(0.10, ability)

    elif difficulty == "Expert":
        return max(0.05, ability - 0.20)

    return ability

In [5]:
print(get_probability(0.80, "Medium"))
print(get_probability(0.80, "Hard"))
print(get_probability(0.80, "Expert"))

0.95
0.8
0.6000000000000001


In [6]:
def generate_user():

    ability = np.random.uniform(0.2,0.95)

    answers = []

    for q in questions:

        p = get_probability(
            ability,
            q["difficulty"]
        )

        correct = np.random.rand() < p

        answers.append(int(correct))

    return ability, answers

In [7]:
ability, answers = generate_user()

print("Ability:",ability)
print(answers)

Ability: 0.4809050891355219
[0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1]


In [8]:
dataset = []

for _ in range(10000):

    ability, answers = generate_user()

    row = {}

    for i, ans in enumerate(answers):
        row[f"Q{i+1}"] = ans

    row["ability"] = ability

    dataset.append(row)

df = pd.DataFrame(dataset)

df.head()

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,ability
0,1,0,0,0,1,1,1,0,0,1,0,0,1,0,1,0.428182
1,0,0,0,1,1,0,0,1,0,1,0,0,0,0,0,0.248789
2,1,0,0,0,0,1,0,1,1,1,1,1,1,0,1,0.610033
3,1,1,0,1,0,0,1,1,0,0,0,0,1,1,1,0.410701
4,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0.847328


In [9]:

categories = {
    "Numerical": ["Q1", "Q2", "Q3"],
    "Logical": ["Q4", "Q5", "Q6"],
    "Pattern": ["Q7", "Q8", "Q9"],
    "Verbal": ["Q10", "Q11"],
    "Analytical": ["Q12", "Q13"],
    "Reflection": ["Q14", "Q15"]
}

In [10]:
def create_features(df):

    feature_df = pd.DataFrame()

    # Copy question answers
    for i in range(1,16):
        feature_df[f"Q{i}"] = df[f"Q{i}"]


    # Total correct answers
    feature_df["total_correct"] = (
        df[[f"Q{i}" for i in range(1,16)]]
        .sum(axis=1)
    )


    # Category scores

    for category, questions_list in categories.items():

        feature_df[f"{category}_score"] = (
            df[questions_list].sum(axis=1)
            /
            len(questions_list)
        )


    return feature_df

In [11]:
features = create_features(df)

features.head()

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q13,Q14,Q15,total_correct,Numerical_score,Logical_score,Pattern_score,Verbal_score,Analytical_score,Reflection_score
0,1,0,0,0,1,1,1,0,0,1,...,1,0,1,7,0.333333,0.666667,0.333333,0.5,0.5,0.5
1,0,0,0,1,1,0,0,1,0,1,...,0,0,0,4,0.000000,0.666667,0.333333,0.5,0.0,0.0
2,1,0,0,0,0,1,0,1,1,1,...,1,0,1,9,0.333333,0.333333,0.666667,1.0,1.0,0.5
3,1,1,0,1,0,0,1,1,0,0,...,1,1,1,8,0.666667,0.333333,0.666667,0.0,0.5,1.0
4,1,1,1,1,1,1,1,0,1,1,...,1,1,1,14,1.000000,1.000000,0.666667,1.0,1.0,1.0


In [12]:
difficulty_weights = {
    "Medium":2,
    "Hard":3,
    "Expert":4
}

In [13]:
question_weights = {}

for q in questions:
    question_weights[f"Q{q['id']}"] = difficulty_weights[q["difficulty"]]

question_weights

{'Q1': 2,
 'Q2': 3,
 'Q3': 4,
 'Q4': 2,
 'Q5': 3,
 'Q6': 3,
 'Q7': 2,
 'Q8': 3,
 'Q9': 4,
 'Q10': 2,
 'Q11': 3,
 'Q12': 3,
 'Q13': 4,
 'Q14': 3,
 'Q15': 2}

In [14]:
def calculate_weighted_score(row):

    score = 0

    for q, weight in question_weights.items():

        if row[q] == 1:
            score += weight

    return score

In [15]:
features["weighted_score"] = features.apply(
    calculate_weighted_score,
    axis=1
)

In [16]:
features.head()

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q14,Q15,total_correct,Numerical_score,Logical_score,Pattern_score,Verbal_score,Analytical_score,Reflection_score,weighted_score
0,1,0,0,0,1,1,1,0,0,1,...,0,1,7,0.333333,0.666667,0.333333,0.5,0.5,0.5,18
1,0,0,0,1,1,0,0,1,0,1,...,0,0,4,0.000000,0.666667,0.333333,0.5,0.0,0.0,10
2,1,0,0,0,0,1,0,1,1,1,...,0,1,9,0.333333,0.333333,0.666667,1.0,1.0,0.5,26
3,1,1,0,1,0,0,1,1,0,0,...,1,1,8,0.666667,0.333333,0.666667,0.0,0.5,1.0,21
4,1,1,1,1,1,1,1,0,1,1,...,1,1,14,1.000000,1.000000,0.666667,1.0,1.0,1.0,40


In [17]:
def assign_level(score):

    if score <= 10:
        return 0

    elif score <=18:
        return 1

    elif score <=26:
        return 2

    elif score <=33:
        return 3

    else:
        return 4

In [18]:
features["cognitive_level"] = (
    features["weighted_score"]
    .apply(assign_level)
)

In [19]:
features["cognitive_level"].value_counts()

,count
cognitive_level,
2,2300
1,2182
3,2162
4,2108
0,1248


In [20]:
final_df = features.copy()

final_df.head()

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,...,Q15,total_correct,Numerical_score,Logical_score,Pattern_score,Verbal_score,Analytical_score,Reflection_score,weighted_score,cognitive_level
0,1,0,0,0,1,1,1,0,0,1,...,1,7,0.333333,0.666667,0.333333,0.5,0.5,0.5,18,1
1,0,0,0,1,1,0,0,1,0,1,...,0,4,0.000000,0.666667,0.333333,0.5,0.0,0.0,10,0
2,1,0,0,0,0,1,0,1,1,1,...,1,9,0.333333,0.333333,0.666667,1.0,1.0,0.5,26,2
3,1,1,0,1,0,0,1,1,0,0,...,1,8,0.666667,0.333333,0.666667,0.0,0.5,1.0,21,2
4,1,1,1,1,1,1,1,0,1,1,...,1,14,1.000000,1.000000,0.666667,1.0,1.0,1.0,40,4


In [21]:
final_df.to_csv(
    "cognitive_dataset.csv",
    index=False
)